# 02 - Layout region 병렬 scheduling과 merge

**학습 목표**: PP-DocLayout-V3가 만든 region을 여러 worker에 배치하고 reading order로 다시 합치는 GLM-OCR의 시스템 구조를 시뮬레이션합니다.

**실행 방법**: Python 3/Jupyter에서 cell을 위에서 아래 순서로 실행합니다. 외부 패키지는 필요하지 않으며 Python 표준 기능만 사용합니다.

실제 wall-clock benchmark가 아닌 deterministic cost model입니다.

In [ ]:
# dict 목록을 사용하면 region의 순서·종류·비용을 이름으로 읽을 수 있어 scheduling 의도가 선명합니다.
regions = [
    {'order': 0, 'kind': 'title', 'cost': 2, 'text': '# Diffusion OCR'},
    {'order': 1, 'kind': 'paragraph', 'cost': 5, 'text': 'OCR reads pixels.'},
    {'order': 2, 'kind': 'table', 'cost': 8, 'text': '|model|TPS|'},
    {'order': 3, 'kind': 'formula', 'cost': 4, 'text': '$p(x|I)$'},
]

def greedy_schedule(items, workers):
    loads = [0] * workers
    assignment = []
    for item in sorted(items, key=lambda x: x['cost'], reverse=True):
        worker = min(range(workers), key=lambda w: loads[w])
        start = loads[worker]
        loads[worker] += item['cost']
        assignment.append((worker, start, loads[worker], item))
    return assignment, max(loads)

serial_cost = sum(region['cost'] for region in regions)
assignment, parallel_cost = greedy_schedule(regions, workers=2)
for worker, start, end, item in assignment:
    print(f"worker={worker} t={start}->{end} {item['kind']}")
merged = '\n\n'.join(r['text'] for r in sorted(regions, key=lambda x: x['order']))
print('serial cost:', serial_cost, 'parallel makespan:', parallel_cost)
print('merged output:\n', merged)
assert merged.startswith('# Diffusion OCR')
assert parallel_cost < serial_cost


실제 pipeline은 crop 크기, batching, GPU memory, detector cost가 추가됩니다. 또한 layout의 `order`가 틀리면 각 crop을 정확히 읽어도 최종 문서가 틀립니다.